In [3]:
# %%
# ============================================================
# PITCHER TALENT FEATURES (v2.2) — expanding-window talent estimate
# Starts from saved pitcher_starts_{SEASON}.csv (NO re-pull).
# Change SEASON, run top to bottom, repeat for the other year.
# ============================================================
import pandas as pd

SEASON = 2026   # <-- change to 2026 and re-run for the test season

DATA_DIR = '/Users/jackdiamond/Documents/Sports_Models/MLB/O/U/Data'
PITCHER_STARTS_IN   = f'{DATA_DIR}/pitcher_starts_{SEASON}.csv'
GAMES_FEATURES_PATH = f'{DATA_DIR}/games_{SEASON}_features.csv'
GAMES_V2_OUT        = f'{DATA_DIR}/games_{SEASON}_features_v2.2.csv'

# %%
# ============================================================
# 1. LOAD saved per-start data + build EXPANDING talent features
#    expanding(min_periods=3): uses ALL starts to date (stable talent),
#    shift(1): leak-free (never sees the current game)
# ============================================================
pitcher_starts = pd.read_csv(PITCHER_STARTS_IN, parse_dates=['game_date'])
pitcher_starts = pitcher_starts.sort_values(['mlbam_id', 'game_date']).reset_index(drop=True)

for stat in ['K_pct', 'BB_pct', 'whiff_pct']:
    pitcher_starts[f'{stat}_talent'] = (
        pitcher_starts.groupby('mlbam_id')[stat]
        .transform(lambda x: x.shift(1).expanding(min_periods=3).mean())
    )

print(pitcher_starts[['pitcher_name', 'game_date', 'K_pct',
                      'K_pct_talent', 'BB_pct_talent', 'whiff_pct_talent']].head(12))

# %%
# ============================================================
# 2. FIX abbreviations (Statcast -> Baseball-Reference)
# ============================================================
statcast_to_bref = {
    'AZ': 'ARI', 'CWS': 'CHW', 'KC': 'KCR', 'SD': 'SDP',
    'SF': 'SFG', 'TB': 'TBR', 'WSH': 'WSN',
}
for col in ['pitcher_team', 'home_team', 'away_team']:
    pitcher_starts[col] = pitcher_starts[col].replace(statcast_to_bref)

print("Pitcher abbrevs:", sorted(pitcher_starts['pitcher_team'].unique()))

# %%
# ============================================================
# 3. JOIN starters onto games
# ============================================================
games = pd.read_csv(GAMES_FEATURES_PATH)
games['game_date'] = pd.to_datetime(games['Date'])

talent_cols = ['K_pct_talent', 'BB_pct_talent', 'whiff_pct_talent']
sp = pitcher_starts[['game_date', 'pitcher_team', 'pitcher_name'] + talent_cols].copy()

# Join 1: HOME starter (pitcher_team == home_team)
home_sp = sp.rename(columns={c: f'home_SP_{c}' for c in talent_cols})
home_sp = home_sp.rename(columns={'pitcher_name': 'home_SP_name'})
games = games.merge(
    home_sp.drop(columns='pitcher_team').assign(_t=home_sp['pitcher_team']),
    left_on=['game_date', 'home_team'], right_on=['game_date', '_t'], how='left'
).drop(columns='_t')

# Join 2: AWAY starter (pitcher_team == away_team)
away_sp = sp.rename(columns={c: f'away_SP_{c}' for c in talent_cols})
away_sp = away_sp.rename(columns={'pitcher_name': 'away_SP_name'})
games = games.merge(
    away_sp.drop(columns='pitcher_team').assign(_t=away_sp['pitcher_team']),
    left_on=['game_date', 'away_team'], right_on=['game_date', '_t'], how='left'
).drop(columns='_t')

# Dedupe doubleheader/residual duplicates
games = games.drop_duplicates(
    subset=['game_date', 'home_team', 'away_team'], keep='first'
).reset_index(drop=True)

print(f"Games: {len(games)}")
print(f"Home SP matched: {games['home_SP_name'].notna().sum()}")
print(f"Away SP matched: {games['away_SP_name'].notna().sum()}")

# %%
# ============================================================
# 4. DROP unmatched + SAVE v2.2 feature set
# ============================================================
before = len(games)
games_v2 = games.dropna(subset=[
    'home_SP_K_pct_talent', 'home_SP_BB_pct_talent', 'home_SP_whiff_pct_talent',
    'away_SP_K_pct_talent', 'away_SP_BB_pct_talent', 'away_SP_whiff_pct_talent'
]).reset_index(drop=True)

print(f"Dropped {before - len(games_v2)} games missing starter data")
print(f"v2.2 dataset ({SEASON}): {len(games_v2)} games")

games_v2.to_csv(GAMES_V2_OUT, index=False)
print(f"Saved -> {GAMES_V2_OUT}")

     pitcher_name  game_date     K_pct  K_pct_talent  BB_pct_talent  \
0    Max Scherzer 2026-03-31  0.181818           NaN            NaN   
1    Max Scherzer 2026-04-06  0.222222           NaN            NaN   
2    Max Scherzer 2026-04-12  0.200000           NaN            NaN   
3    Max Scherzer 2026-04-18  0.000000      0.201347       0.096633   
4    Max Scherzer 2026-04-24  0.000000      0.151010       0.083838   
5    Max Scherzer 2026-06-10  0.222222      0.120808       0.104571   
6   Jose Quintana 2026-03-29  0.105263           NaN            NaN   
7   Jose Quintana 2026-04-15  0.055556           NaN            NaN   
8   Jose Quintana 2026-04-20  0.040000           NaN            NaN   
9   Jose Quintana 2026-04-26  0.250000      0.066940       0.157583   
10  Jose Quintana 2026-05-01  0.130435      0.112705       0.143187   
11  Jose Quintana 2026-05-07  0.090909      0.116251       0.114550   

    whiff_pct_talent  
0                NaN  
1                NaN  
2      